In [46]:
import html
import os
from typing import List
import mdtraj as md

class Py3DmolContent:
    def __init__(self, pdb_paths: List[str], align: bool = True) -> None:
        self.pdb_paths = pdb_paths
        self.align = align

        self.trajectories = [
            md.load(path, top=path) for path in self.pdb_paths
        ]

        if self.align and len(self.trajectories) > 1:
            self._align_structures()

    def _align_structures(self) -> None:
        ref_traj = self.trajectories[0]
        ref_ca = ref_traj.topology.select("name CA")

        for target_traj in self.trajectories[1:]:
            target_ca = target_traj.topology.select("name CA")
            if len(ref_ca) != len(target_ca):
                ref_ca = ref_traj.topology.select("backbone")
                target_ca = target_traj.topology.select("backbone")

            target_traj.superpose(ref_traj, atom_indices=target_ca, ref_atom_indices=ref_ca)

    def render_html(self) -> str:
        colors = ["cyan", "magenta", "orange", "green", "yellow", "red"]
        js_models = ""

        for idx, traj in enumerate(self.trajectories):
            if self.align and len(self.trajectories) > 1:
                tmp_filename = f"_temp_aligned_{idx}.pdb"
                traj.save_pdb(tmp_filename)
                with open(tmp_filename, "r") as f:
                    pdb_str = f.read()
                if os.path.exists(tmp_filename):
                    os.remove(tmp_filename)
            else:
                with open(self.pdb_paths[idx], "r") as f:
                    pdb_str = f.read()

            safe_pdb = pdb_str.replace("\\", "\\\\").replace("`", "\\`").replace("$", "\\$")
            color = "spectrum" if len(self.trajectories) == 1 else colors[idx % len(colors)]
            js_models += f'v.addModel(`{safe_pdb}`, "pdb"); v.setStyle({{model: {idx}}}, {{cartoon: {{color: "{color}"}}}});\n'

        # Complete standalone document inside the iframe
        inner_html = f"""<!DOCTYPE html>
            <html>
            <head>
                <script src="https://3Dmol.org/build/3Dmol-min.js"></script>
                <style>
                    html, body {{ margin: 0; padding: 0; width: 100%; height: 100%; overflow: hidden; }}
                    #gviewer {{ width: 100%; height: 100%; position: absolute; }}
                </style>
            </head>
            <body>
                <div id="gviewer"></div>
                <script>
                    document.addEventListener("DOMContentLoaded", function() {{
                        let elem = document.getElementById("gviewer");
                        let v = $3Dmol.createViewer(elem, {{backgroundColor: "white"}});
                        {js_models}
                        v.zoomTo();
                        v.render();
                    }});
                </script>
            </body>
            </html>"""

        # Escape HTML attributes safely for iframe srcdoc injection
        escaped_srcdoc = html.escape(inner_html, quote=True)

        return f"""
        <div style="width: 100%; height: 500px; text-align: center;">
            <iframe 
                srcdoc="{escaped_srcdoc}" 
                width="100%" 
                height="500px" 
                style="border: none; border-radius: 8px; width: 100%; height: 500px;">
            </iframe>
        </div>
        """


In [47]:
pdb_dir = "pdb"
pdb_root = "Ab42_seq"
pdb_no = 1000

#######

pdb_name = f"{pdb_root}_{pdb_no}.pdb"
pdb_path = os.path.join(pdb_dir, pdb_name)

ngl_view = Py3DmolContent(
    pdb_paths=[pdb_path]
)

slide = SingleColumnSlide(
    title="Protein Structure",
    content_block=ngl_view
)

slide.display()

In [48]:
pdb_dir = "pdb"
pdb_root = "Ab42_seq"
pdb_no = 1000

#######

pdb_paths = [
    os.path.join(pdb_dir, i) for i in os.listdir(pdb_dir)
]

ngl_view = Py3DmolContent(
    pdb_paths=pdb_paths
)

slide = SingleColumnSlide(
    title="Protein Structure",
    content_block=ngl_view
)

slide.display()

In [49]:
!jupyter nbconvert py3dmoltest.ipynb --to slides --no-input --output py3dmoltest
!brave py3dmoltest.slides.html

[NbConvertApp] Converting notebook py3dmoltest.ipynb to slides
[NbConvertApp] Writing 350163 bytes to py3dmoltest.slides.html
Opening in existing browser session.
